# 1 · Function Calling / Tool Use
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟢 Básica**   |   **Dependencias: solo `anthropic`**

Este es el notebook más simple de la serie — ideal para empezar si tuviste problemas de instalación con paquetes más pesados (LangChain, LlamaIndex, MCP). Aquí solo necesitas el SDK oficial de Anthropic.

**Qué vas a construir:** el ciclo básico `request → tool_use → tool_result → respuesta final`, tal como se explicó en la charla, con **una sola vuelta** (en el notebook 2 lo convertimos en un loop completo).


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar por qué un LLM no puede ejecutar acciones por sí solo, y qué problema resuelve *function calling / tool use*.
- Definir el schema de una herramienta (`name`, `description`, `parameters`) para que el modelo la invoque correctamente.
- Implementar el ciclo completo de una sola vuelta: enviar la herramienta → recibir la solicitud del modelo → ejecutar la función real → devolver el resultado → obtener la respuesta final.


## 📚 Teoría: Function Calling / Tool Use

Un LLM, por sí solo, **solo genera texto**. No puede consultar una base de datos, llamar a una API, ni saber si el clima de hoy cambió — su conocimiento es estático y no tiene manos para actuar en el mundo real.

*Function calling* (también llamado *tool use*) es el mecanismo que resuelve esto: tú le describes al modelo, mediante un **schema estructurado**, qué funciones existen y qué parámetros reciben. El modelo **decide** cuándo necesita usar una, y en vez de responder texto libre, responde una estructura indicando qué función llamar y con qué argumentos. Nunca ejecuta la función él mismo — eso siempre es responsabilidad de tu código.

El ciclo tiene 4 pasos:
1. Tu app envía el prompt + la definición de herramientas al modelo.
2. El modelo responde pidiendo ejecutar una función (o responde texto normal si no la necesita).
3. Tu código ejecuta la función real.
4. Envías el resultado de vuelta al modelo, que genera la respuesta final para el usuario.

**El detalle que más importa:** la calidad de la `description` de cada herramienta es lo que determina si el modelo la usa en el momento correcto y con los parámetros correctos — es, en la práctica, el "manual de instrucciones" que el modelo lee para decidir.


## 0. Instalación (única dependencia)

In [ ]:
!pip install -q anthropic

### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. Definir una herramienta (schema)

Una herramienta se define con: `name`, `description` (crítico: el modelo decide **cuándo** usarla en base a esto) y `input_schema` (JSON Schema de los parámetros).


In [ ]:
import anthropic

client = anthropic.Anthropic()

# Herramienta simple: consultar el clima de una ciudad (simulado)
tools = [
    {
        "name": "get_weather",
        "description": "Obtiene el clima actual de una ciudad. Úsala cuando el usuario pregunte por el clima o temperatura de un lugar.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nombre de la ciudad, ej: Cali, Bogotá"}
            },
            "required": ["city"]
        }
    }
]

# Implementación real de la herramienta (aquí simulada; en producción llamarías a una API real)
def get_weather(city: str) -> str:
    fake_db = {"cali": "28°C, soleado", "bogota": "16°C, nublado", "medellin": "23°C, parcialmente nublado"}
    return fake_db.get(city.lower(), f"No tengo datos de clima para {city}")


## 2. Primer turno: el modelo decide si necesita la herramienta

Le enviamos un mensaje que **requiere** usar la herramienta. El modelo no responde texto directamente: responde un bloque `tool_use`.


In [ ]:
messages = [{"role": "user", "content": "¿Qué clima hace hoy en Cali?"}]

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("Razón de parada:", response.stop_reason)  # debería ser "tool_use"
for block in response.content:
    print(block.type, "->", block)


## 3. Ejecutar la herramienta y devolver el resultado

Cuando `stop_reason == "tool_use"`, tu código debe: (1) ejecutar la función real, (2) enviar el resultado de vuelta como `tool_result`, (3) dejar que el modelo continúe.


In [ ]:
messages.append({"role": "assistant", "content": response.content})

tool_results = []
for block in response.content:
    if block.type == "tool_use":
        if block.name == "get_weather":
            result = get_weather(**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result
            })

messages.append({"role": "user", "content": tool_results})

# Segundo turno: el modelo ya tiene el resultado y da la respuesta final
final_response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print(final_response.content[0].text)


## 🧪 Ejercicio

Agrega una segunda herramienta `convert_currency(amount, from_currency, to_currency)` (puedes simular tasas de cambio fijas) y haz una pregunta que combine ambas herramientas, por ejemplo: *"¿Qué clima hace en Bogotá y cuánto son 50 USD en COP?"*. Observa cuántos bloques `tool_use` genera el modelo en un mismo turno.

---
**Siguiente notebook:** `agente_react_manual.ipynb` — convertimos esto en un loop completo (el ciclo ReAct de la charla).


In [ ]:
# Tu código aquí
